# Hearo - YAMNet Fine-Tuning (v2)
청각장애인을 위한 가정용 환경음 인식 모델 파인튜닝

## 11개 분류 카테고리
1. 가전제품
2. 개 짖음
3. 노크 소리
4. 도어락
5. 물 소리
6. 사이렌
7. 아기 울음
8. 인터폰
9. 초인종
10. 창문 깨지는 소리
11. 휴대폰 벨소리

## v2 변경사항
- 6종 → 11종 카테고리 확장
- Data Leakage 해결: 원본 단위 Train/Val 분리 후 Train만 증강
- 원본 307개 → 증강 후 학습

## 1. 환경 설정

In [ ]:
!pip install tensorflow tensorflow_hub resampy soundfile pydub matplotlib seaborn scikit-learn

# Colab 한글 폰트 설치
!apt-get install -y fonts-nanum > /dev/null 2>&1

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 핵심: addfont()로 폰트를 직접 등록 → rcParams가 확실히 인식
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)

# 전역 기본 폰트로 설정 (모든 그래프에 자동 적용)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 테스트
fig, ax = plt.subplots(figsize=(6, 1.5))
ax.text(0.5, 0.5, '가전제품 개 짖음 초인종 노크 소리 테스트',
        ha='center', va='center', fontsize=14)
ax.axis('off')
plt.show()

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import os
import csv
import resampy
import soundfile as sf
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from pydub import AudioSegment
import io
import random

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
print(f"기본 폰트: {plt.rcParams['font.family']}")

## 2. Google Drive 마운트 & 데이터 업로드

**사용법:** `소리 정리` 폴더를 Google Drive에 업로드한 뒤 실행하세요.

Google Drive 구조:
```
내 드라이브/
  소리 정리/
    가전제품/
    개 짖음/
    노크 소리/
    도어락/
    물 소리/
    사이렌/
    아기 울음/
    인터폰/
    초인종/
    창문 깨지는 소리/
    휴대폰 벨소리/
```

In [ ]:
from google.colab import drive
import unicodedata

drive.mount('/content/drive')

# 데이터 경로 설정
DATA_DIR = '/content/drive/MyDrive/소리 정리'

# 지원 확장자
AUDIO_EXT = ('.wav', '.mp3', '.flac', '.m4a')

# Google Drive는 한글을 NFD(분해형)로 저장하므로 NFC(조합형)로 변환
# 원본 폴더 이름(NFD) → 표시용 이름(NFC) 매핑
_raw_dirs = sorted(os.listdir(DATA_DIR))
CATEGORIES = [unicodedata.normalize('NFC', d) for d in _raw_dirs]
_cat_to_raw = {unicodedata.normalize('NFC', d): d for d in _raw_dirs}
NUM_CLASSES = len(CATEGORIES)

def get_cat_path(category):
    """카테고리 이름으로 실제 폴더 경로 반환 (NFD 호환)"""
    return os.path.join(DATA_DIR, _cat_to_raw[category])

print(f"카테고리 ({NUM_CLASSES}개): {CATEGORIES}")
for cat in CATEGORIES:
    cat_path = get_cat_path(cat)
    files = [f for f in os.listdir(cat_path) if f.endswith(AUDIO_EXT)]
    print(f"  {cat}: {len(files)}개")

## 3. YAMNet 모델 로드

In [ ]:
print("YAMNet 모델 로드 중...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')
print("YAMNet 모델 로드 완료!")

## 4. 오디오 로드 & 전처리 함수

In [ ]:
TARGET_SR = 16000  # YAMNet 요구 샘플레이트

def load_audio(file_path):
    """wav/mp3/m4a/flac 파일을 16kHz mono float32로 로드"""
    ext = os.path.splitext(file_path)[1].lower()

    if ext in ('.mp3', '.m4a'):
        # mp3, m4a -> wav 변환 (pydub 사용)
        audio = AudioSegment.from_file(file_path)
        audio = audio.set_channels(1).set_frame_rate(TARGET_SR)
        samples = np.array(audio.get_array_of_samples(), dtype=np.float32) / 32768.0
        return samples
    else:
        # wav, flac 등
        wav_data, sample_rate = sf.read(file_path, dtype='float32')
        if len(wav_data.shape) > 1:
            wav_data = wav_data.mean(axis=1)  # stereo -> mono
        if sample_rate != TARGET_SR:
            wav_data = resampy.resample(wav_data, sample_rate, TARGET_SR)
        return wav_data

# 테스트
test_cat = CATEGORIES[0]
test_dir = get_cat_path(test_cat)
test_file = [f for f in os.listdir(test_dir) if f.endswith(AUDIO_EXT)][0]
test_audio = load_audio(os.path.join(test_dir, test_file))
print(f"테스트 로드: {test_cat}/{test_file}")
print(f"  길이: {len(test_audio)/TARGET_SR:.2f}초, shape: {test_audio.shape}")

## 5. 데이터 증강 (Data Augmentation)
307개 원본 샘플을 증강하여 학습 효과 향상

**v2 변경:** Train 데이터만 증강 (Data Leakage 방지)

In [ ]:
def augment_audio(waveform, sr=TARGET_SR):
    """하나의 오디오에서 여러 증강 버전 생성"""
    augmented = []

    # 1. 원본
    augmented.append(waveform.copy())

    # 2. 가우시안 노이즈 추가 (약한)
    noise = np.random.normal(0, 0.005, len(waveform)).astype(np.float32)
    augmented.append(waveform + noise)

    # 3. 가우시안 노이즈 추가 (강한)
    noise = np.random.normal(0, 0.015, len(waveform)).astype(np.float32)
    augmented.append(waveform + noise)

    # 4. 볼륨 변경 (작게)
    augmented.append(waveform * 0.6)

    # 5. 볼륨 변경 (크게)
    augmented.append(np.clip(waveform * 1.4, -1.0, 1.0))

    # 6. Time Shift (앞뒤로 밀기)
    shift = int(sr * 0.2)  # 0.2초
    shifted = np.roll(waveform, shift)
    augmented.append(shifted)

    # 7. Time Shift (반대 방향)
    shifted = np.roll(waveform, -shift)
    augmented.append(shifted)

    # 8. Pitch Shift (속도 변경으로 근사) - 빠르게
    fast = resampy.resample(waveform, sr, int(sr * 1.1))
    if len(fast) > len(waveform):
        fast = fast[:len(waveform)]
    else:
        fast = np.pad(fast, (0, len(waveform) - len(fast)))
    augmented.append(fast)

    # 9. Pitch Shift - 느리게
    slow = resampy.resample(waveform, sr, int(sr * 0.9))
    if len(slow) > len(waveform):
        slow = slow[:len(waveform)]
    else:
        slow = np.pad(slow, (0, len(waveform) - len(slow)))
    augmented.append(slow)

    # 10. 랜덤 구간 마스킹
    masked = waveform.copy()
    mask_len = int(sr * 0.1)  # 0.1초 구간
    mask_start = random.randint(0, max(0, len(masked) - mask_len))
    masked[mask_start:mask_start + mask_len] = 0
    augmented.append(masked)

    return augmented

print(f"증강 함수 준비 완료! (원본 1개 -> {len(augment_audio(np.zeros(16000)))}개)")

## 6. 원본 단위 Train/Val 분리 (Data Leakage 방지)

**핵심:** 증강 전에 원본 파일 단위로 먼저 분리합니다.
같은 원본의 변형이 Train과 Val에 동시에 들어가는 것을 방지합니다.

In [ ]:
def extract_embedding(waveform):
    """YAMNet에서 1024차원 임베딩 벡터 추출"""
    scores, embeddings, spectrogram = yamnet_model(waveform)
    return embeddings.numpy().mean(axis=0)

# ===== 1단계: 원본 파일 목록을 카테고리별로 수집 =====
original_files = []  # (file_path, label_idx) 리스트

for label_idx, category in enumerate(CATEGORIES):
    cat_path = get_cat_path(category)
    audio_files = [f for f in os.listdir(cat_path) if f.endswith(AUDIO_EXT)]
    for fname in audio_files:
        original_files.append((os.path.join(cat_path, fname), label_idx))

print(f"전체 원본 파일 수: {len(original_files)}")

# ===== 2단계: 원본 단위로 Train/Val 분리 (80:20) =====
file_paths = [f[0] for f in original_files]
file_labels = [f[1] for f in original_files]

train_files, val_files, train_labels, val_labels = train_test_split(
    file_paths, file_labels, test_size=0.2, random_state=42, stratify=file_labels
)

print(f"Train 원본: {len(train_files)}개")
print(f"Val 원본: {len(val_files)}개")

# 카테고리별 분포 확인
print(f"\n카테고리별 Train/Val 분포:")
for i, cat in enumerate(CATEGORIES):
    tr = sum(1 for l in train_labels if l == i)
    va = sum(1 for l in val_labels if l == i)
    print(f"  {cat}: Train {tr}개, Val {va}개")

# ===== 3단계: Train은 증강, Val은 원본만 사용 =====
X_train, y_train = [], []
X_val, y_val = [], []

# Train: 증강 적용
print(f"\n--- Train 데이터 증강 중 ---")
for fpath, label in zip(train_files, train_labels):
    try:
        waveform = load_audio(fpath)
        if len(waveform) < TARGET_SR:
            waveform = np.pad(waveform, (0, TARGET_SR - len(waveform)))

        augmented_list = augment_audio(waveform)
        for aug_waveform in augmented_list:
            embedding = extract_embedding(aug_waveform)
            X_train.append(embedding)
            y_train.append(label)
    except Exception as e:
        print(f"  [오류] {os.path.basename(fpath)}: {e}")

# Val: 원본만 사용 (증강 없음)
print(f"\n--- Val 데이터 (원본만) ---")
for fpath, label in zip(val_files, val_labels):
    try:
        waveform = load_audio(fpath)
        if len(waveform) < TARGET_SR:
            waveform = np.pad(waveform, (0, TARGET_SR - len(waveform)))

        embedding = extract_embedding(waveform)
        X_val.append(embedding)
        y_val.append(label)
    except Exception as e:
        print(f"  [오류] {os.path.basename(fpath)}: {e}")

X_train = np.array(X_train)
y_train = np.array(y_train)
X_val = np.array(X_val)
y_val = np.array(y_val)

print(f"\n===== 데이터 준비 완료 (Data Leakage 방지) =====")
print(f"Train 샘플 수: {len(X_train)} (원본 {len(train_files)}개 x 10 증강)")
print(f"Val 샘플 수: {len(X_val)} (원본만, 증강 없음)")
print(f"임베딩 차원: {X_train.shape[1]}")

## 7. 분류 모델 구성 & 학습
YAMNet 임베딩(1024차원) 위에 Dense 레이어 추가

**v2:** Train/Val 분리가 이미 완료되었으므로 바로 학습

In [ ]:
# v2: 이미 분리된 X_train, X_val 사용
print(f"Train: {X_train.shape[0]}개, Val: {X_val.shape[0]}개")
print(f"클래스 수: {NUM_CLASSES}")
print("Data Leakage 방지: 원본 단위 분리 완료")

## 8. 분류 모델 구성 & 학습
YAMNet 임베딩(1024차원) 위에 Dense 레이어 추가

In [ ]:
# 모델 구성
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024,)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# 학습 (v2: X_val, y_val 직접 사용)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=15,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5
        )
    ]
)

print(f"\n최종 검증 정확도: {max(history.history['val_accuracy']):.4f}")

## 9. 학습 결과 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 정확도 그래프
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# 손실 그래프
axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 10. 혼동 행렬 (Confusion Matrix) & 분류 리포트

In [ ]:
# 예측 (v2: X_val, y_val 사용)
y_pred = model.predict(X_val)
y_pred_classes = np.argmax(y_pred, axis=1)

# 혼동 행렬
cm = confusion_matrix(y_val, y_pred_classes)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CATEGORIES, yticklabels=CATEGORIES, ax=ax)
ax.set_title('Confusion Matrix', fontsize=18)
ax.set_xlabel('Predicted', fontsize=14)
ax.set_ylabel('Actual', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# 분류 리포트
print("\n===== Classification Report =====")
print(classification_report(y_val, y_pred_classes, target_names=CATEGORIES))

## PPT 발표용 시각화

학습 결과를 발표 자료에 삽입할 수 있도록 고해상도 이미지로 저장합니다.
저장 경로: `Google Drive/Hearo_model/figures/`

## 11. 모델 저장

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/Hearo_model'
FIG_DIR = os.path.join(SAVE_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({
    'axes.unicode_minus': False,
    'font.size': 14,
    'axes.titlesize': 18,
    'axes.labelsize': 15,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 13,
    'figure.dpi': 200,
})

# ===== 1. 학습 곡선 (Accuracy & Loss) =====
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

epochs = range(1, len(history.history['accuracy']) + 1)

axes[0].plot(epochs, history.history['accuracy'], 'b-o', markersize=3, label='Train')
axes[0].plot(epochs, history.history['val_accuracy'], 'r-o', markersize=3, label='Validation')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.05])

axes[1].plot(epochs, history.history['loss'], 'b-o', markersize=3, label='Train')
axes[1].plot(epochs, history.history['val_loss'], 'r-o', markersize=3, label='Validation')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '1_learning_curve.png'), bbox_inches='tight', facecolor='white')
plt.show()
print(f"저장: {FIG_DIR}/1_learning_curve.png")

In [ ]:
# Keras 모델 저장
SAVE_DIR = '/content/drive/MyDrive/Hearo_model'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save(os.path.join(SAVE_DIR, 'hearo_classifier.keras'))

# 카테고리 이름 저장
with open(os.path.join(SAVE_DIR, 'categories.txt'), 'w', encoding='utf-8') as f:
    for cat in CATEGORIES:
        f.write(cat + '\n')

print(f"모델 저장 완료: {SAVE_DIR}")
print(f"  - hearo_classifier.keras")
print(f"  - categories.txt")

In [ ]:
# ===== 2. 혼동 행렬 (PPT용 고해상도) =====
y_pred = model.predict(X_val)
y_pred_classes = np.argmax(y_pred, axis=1)
cm = confusion_matrix(y_val, y_pred_classes)

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CATEGORIES, yticklabels=CATEGORIES,
            annot_kws={'size': 14}, linewidths=0.5, ax=ax)
ax.set_title('Confusion Matrix', pad=20, fontsize=20)
ax.set_xlabel('Predicted Label', fontsize=15)
ax.set_ylabel('Actual Label', fontsize=15)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(rotation=0, fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '2_confusion_matrix.png'), bbox_inches='tight', facecolor='white')
plt.show()
print(f"저장: {FIG_DIR}/2_confusion_matrix.png")

## 12. TFLite 변환 (Raspberry Pi 배포용)

In [ ]:
# ===== 3. 카테고리별 F1-Score 막대 그래프 =====
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_val, y_pred_classes, labels=range(NUM_CLASSES)
)

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(NUM_CLASSES)
width = 0.25

bars1 = ax.bar(x - width, precision, width, label='Precision', color='#4472C4')
bars2 = ax.bar(x, recall, width, label='Recall', color='#ED7D31')
bars3 = ax.bar(x + width, f1, width, label='F1-Score', color='#70AD47')

ax.set_ylabel('Score')
ax.set_title('Per-Class Precision / Recall / F1-Score')
ax.set_xticks(x)
ax.set_xticklabels(CATEGORIES, rotation=45, ha='right')
ax.set_ylim([0, 1.15])
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.annotate(f'{height:.2f}',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '3_per_class_f1.png'), bbox_inches='tight', facecolor='white')
plt.show()
print(f"저장: {FIG_DIR}/3_per_class_f1.png")

In [ ]:
# TFLite 변환
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(SAVE_DIR, 'hearo_classifier.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"TFLite 모델 저장: {tflite_path}")
print(f"모델 크기: {len(tflite_model) / 1024:.1f} KB")

In [ ]:
# ===== 4. 카테고리별 데이터 분포 (Train vs Val) =====
fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(NUM_CLASSES)
width = 0.35

train_counts = [sum(1 for l in train_labels if l == i) for i in range(NUM_CLASSES)]
val_counts = [sum(1 for l in val_labels if l == i) for i in range(NUM_CLASSES)]
train_aug_counts = [c * 10 for c in train_counts]

bars1 = ax.bar(x - width/2, train_aug_counts, width, label=f'Train (augmented, total {sum(train_aug_counts)})', color='#4472C4')
bars2 = ax.bar(x + width/2, val_counts, width, label=f'Val (original only, total {sum(val_counts)})', color='#ED7D31')

ax.set_ylabel('Number of Samples')
ax.set_title('Data Distribution: Train (Augmented) vs Validation (Original)')
ax.set_xticks(x)
ax.set_xticklabels(CATEGORIES, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.annotate(f'{int(bar.get_height())}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.annotate(f'{int(bar.get_height())}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '4_data_distribution.png'), bbox_inches='tight', facecolor='white')
plt.show()
print(f"저장: {FIG_DIR}/4_data_distribution.png")

## 13. 새 소리 테스트 (Threshold 적용)

**Threshold(임계값):** 모델의 예측 확률이 설정한 기준 이상일 때만 알림을 보냅니다.
- 확률 >= threshold → 해당 소리로 판정 → 알림
- 확률 < threshold → 확신 불가 → 무시 (알림 없음)

이를 통해 학습하지 않은 소리(TV, 말소리, 자동차 등)에 대한 오알림을 방지합니다.

In [ ]:
THRESHOLD = 0.9  # 90% 이상일 때만 알림

# ===== 5. 모델 구조 요약 테이블 (PPT 텍스트용) =====
total_original = len(original_files)
total_train_original = len(train_files)
total_val_original = len(val_files)
total_train_augmented = len(X_train)
best_val_acc = max(history.history['val_accuracy'])
best_epoch = history.history['val_accuracy'].index(best_val_acc) + 1
total_epochs = len(history.history['accuracy'])

print("=" * 55)
print("        Hearo YAMNet Fine-Tuning 학습 결과 요약")
print("=" * 55)
print(f"  카테고리 수:         {NUM_CLASSES}종")
print(f"  원본 데이터 수:      {total_original}개")
print(f"  Train 원본:          {total_train_original}개")
print(f"  Train 증강 후:       {total_train_augmented}개 (x10)")
print(f"  Val (원본만):        {total_val_original}개")
print(f"  증강 배율:           10x (10가지 기법)")
print(f"  ---")
print(f"  총 Epoch 수:         {total_epochs}")
print(f"  Best Epoch:          {best_epoch}")
print(f"  최종 검증 정확도:    {best_val_acc*100:.1f}%")
print(f"  Threshold:           {THRESHOLD*100:.0f}%")
print("=" * 55)
print()
print(">>> 논문 빈칸 채우기 값:")
print(f"    원본 데이터 수: {total_original}개")
print(f"    증강 후 학습 데이터: {total_train_augmented}개")
print(f"    검증 정확도: {best_val_acc*100:.1f}%")

In [ ]:
def predict_sound(file_path, threshold=THRESHOLD):
    """파인튜닝된 모델로 소리 분류 (threshold 적용)"""
    waveform = load_audio(file_path)
    if len(waveform) < TARGET_SR:
        waveform = np.pad(waveform, (0, TARGET_SR - len(waveform)))

    embedding = extract_embedding(waveform)
    embedding = embedding.reshape(1, -1)

    prediction = model.predict(embedding, verbose=0)[0]

    sorted_idx = np.argsort(prediction)[::-1]
    top_category = CATEGORIES[sorted_idx[0]]
    top_confidence = prediction[sorted_idx[0]]

    print(f"\n파일: {os.path.basename(file_path)}")
    print(f"길이: {len(waveform)/TARGET_SR:.1f}초")
    print(f"{'='*40}")

    for idx in sorted_idx:
        bar = '#' * int(prediction[idx] * 30)
        marker = ' <<' if idx == sorted_idx[0] else ''
        print(f"  {CATEGORIES[idx]:<10s} {prediction[idx]*100:5.1f}% {bar}{marker}")

    # Threshold 판정
    print(f"\n  Threshold: {threshold*100:.0f}%")
    if top_confidence >= threshold:
        print(f"  --> [{top_category}] 감지! ({top_confidence*100:.1f}%) -> 알림 전송")
    else:
        print(f"  --> 확신 불가 ({top_confidence*100:.1f}%) -> 무시 (알림 없음)")

    return top_category, top_confidence

print(f"predict_sound() 함수 준비 완료! (Threshold: {THRESHOLD*100:.0f}%)")
print("사용법: predict_sound('/path/to/audio.wav')")

In [ ]:
# 각 카테고리에서 1개씩 테스트
print("===== 카테고리별 샘플 테스트 (Threshold 적용) =====")

results = []
for category in CATEGORIES:
    cat_path = get_cat_path(category)
    audio_files = [f for f in os.listdir(cat_path) if f.endswith(AUDIO_EXT)]
    if audio_files:
        test_file = os.path.join(cat_path, audio_files[0])
        predicted, confidence = predict_sound(test_file)
        correct = '  O' if predicted == category else '  X'
        detected = 'O' if confidence >= THRESHOLD else 'X'
        results.append((category, predicted, confidence, correct, detected))

print(f"\n\n===== 테스트 요약 =====")
print(f"Threshold: {THRESHOLD*100:.0f}%")
correct_count = sum(1 for r in results if r[3].strip() == 'O')
detected_count = sum(1 for r in results if r[4] == 'O')
print(f"분류 정확도: {correct_count}/{len(results)} ({correct_count/len(results)*100:.0f}%)")
print(f"알림 발생: {detected_count}/{len(results)}개")
for actual, predicted, conf, mark, det in results:
    alert = "알림" if det == 'O' else "무시"
    print(f"  {mark} {actual} -> {predicted} ({conf*100:.1f}%) [{alert}]")

## 14. 파일 업로드로 직접 테스트

In [ ]:
from google.colab import files
from IPython.display import Audio, display

print(f"소리 파일을 업로드하세요 (wav, mp3)")
print(f"Threshold: {THRESHOLD*100:.0f}% 이상일 때만 알림 발생\n")
uploaded = files.upload()

for filename in uploaded:
    waveform = load_audio(filename)
    display(Audio(waveform, rate=TARGET_SR))
    predicted, confidence = predict_sound(filename)
    print()